In [30]:
import osmnx as ox
import networkx as nx
import folium
import numpy as np
from datetime import datetime, timedelta
from folium.plugins import TimestampedGeoJson

place = "District 1, Ho Chi Minh City, Vietnam"
G = ox.graph_from_place(place, network_type='drive')

hubs = [
    (10.780, 106.695),
    (10.775, 106.700),
    (10.770, 106.705)
]

customers = [
    (10.782, 106.705),
    (10.768, 106.698),
    (10.772, 106.690)
]

def simulate(start, end, start_time):
    s = ox.distance.nearest_nodes(G, start[1], start[0])
    e = ox.distance.nearest_nodes(G, end[1], end[0])

    path = nx.shortest_path(G, s, e, weight='length')
    coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path]

    idx = np.linspace(0, len(coords)-1, 10).astype(int)
    coords = [coords[i] for i in idx]

    features = []
    for i,(lat,lon) in enumerate(coords):
        t = (start_time + timedelta(seconds=i*20)).isoformat()
        features.append({
            "type":"Feature",
            "geometry":{"type":"Point","coordinates":[lon,lat]},
            "properties":{"time":t}
        })
    return features, coords

m = folium.Map(location=[10.775,106.700], zoom_start=14)

all_f = []
base = datetime(2026,1,1,8,0,0)

for i in range(3):
    f, route = simulate(hubs[i], customers[i], base)
    all_f += f
    folium.PolyLine(route).add_to(m)

TimestampedGeoJson(
    {"type":"FeatureCollection","features":all_f},
    period="PT20S"
).add_to(m)

m

**Nhận xét:**

Hệ thống mô phỏng xe di chuyển trên mạng đường bằng cách tìm đường ngắn nhất giữa điểm xuất phát và điểm đến.

Lộ trình được chia thành nhiều bước nhỏ theo thời gian, mỗi bước tương ứng với một vị trí của xe.
Nhờ đó có thể quan sát quá trình di chuyển liên tục trên bản đồ.

Mô hình giúp hiểu cách xe được điều phối và cập nhật trạng thái theo thời gian trong hệ thống thực tế.